In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
pd.set_option('display.max_rows', 700)
pd.set_option('display.max_columns', 700)

# Fast 3D ResNet Training (Step by Step)


This notebook runs the training flow step-by-step using the existing project functions.

In [2]:
import os
import sys
from pathlib import Path

# Resolve paths so imports work regardless of notebook launch folder
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
app_dir = repo_root / "app"

if str(app_dir) not in sys.path:
    sys.path.insert(0, str(app_dir))

# Keep config relative paths consistent with CLI usage
os.chdir(app_dir)

print(f"Repo root: {repo_root}")
print(f"App dir:   {app_dir}")
print(f"CWD:       {Path.cwd().resolve()}")

Repo root: /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model
App dir:   /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/app
CWD:       /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/app


In [3]:
import torch

from src.config import Configuration
from src.data import UPennGBMDataset
from src.models import FastResNet3DLightningModule
from scripts import train_fast_resnet, test_fast_resnet

print("Imports OK")
print("CUDA available:", torch.cuda.is_available())

Imports OK
CUDA available: True


In [4]:
config = Configuration()

print("mr_data:", config.mr_data)
print("tensor_dir:", config.mr_nf_tensors)
print("num_classes:", len(config.labels))
print("labels:", config.labels)

assert os.path.exists(config.mr_data), f"Missing clinical CSV: {config.mr_data}"
assert os.path.exists(config.mr_nf_tensors), f"Missing tensor folder: {config.mr_nf_tensors}"
print("Config and paths look good.")

mr_data: ../data/MR/metadata/UPENN-GBM_clinical_info_v2.1.csv
tensor_dir: ../data/MR_NIfTI/images_tensors
num_classes: 2
labels: [0, 1]
Config and paths look good.


In [5]:
train_ds = UPennGBMDataset(config, partition="train")
val_ds = UPennGBMDataset(config, partition="val")
test_ds = UPennGBMDataset(config, partition="test")

print("train size:", len(train_ds))
print("val size:", len(val_ds))
print("test size:", len(test_ds))

sample = train_ds[0]
print("image shape:", tuple(sample["image"].shape))
print("tabular shape:", tuple(sample["tabular"].shape))
print("label:", int(sample["label"]))

train size: 501
val size: 51
test size: 51
image shape: (4, 240, 240, 155)
tabular shape: (14,)
label: 0


In [6]:
# Pretrained 3D ResNet sanity check on a single sample
num_classes = len(config.labels)
pl_model = FastResNet3DLightningModule(
    num_classes=num_classes,
    learning_rate=1e-3,
    input_size=96,
    use_pretrained_backbone=True,
    freeze_backbone=False,
 )

x = sample["image"].unsqueeze(0)  # (1, C, D, H, W)
with torch.no_grad():
    logits = pl_model(x)

print("logits shape:", tuple(logits.shape))
print("expected shape:", (1, num_classes))

logits shape: (1, 2)
expected shape: (1, 2)


In [10]:
# Training with pretrained torchvision r3d_18 backbone
trained_model = train_fast_resnet(
    config=config,
    epochs=10,
    batch_size=2,
    learning_rate=1e-3,
    num_workers=2,
    input_size=96,
    base_channels=16,
    dropout=0.3,
    label_smoothing=0.0,
    weight_decay=1e-4,
    use_binary_bce=True,
    use_amp=False,  # safer when loss becomes NaN
    use_pretrained_backbone=True,
    freeze_backbone=False,
    enable_early_stopping=True,
    early_stopping_patience=4,s
    early_stopping_min_delta=1e-4,
    gradient_clip_val=1.0,
 )

trained_model

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model     | VideoResNet       | 33.2 M | train | 0    
1 | criterion | BCEWithLogitsLoss | 0      | train | 0    
----------------------------------------------------------------
33.2 M    Trainable params
0         Non-trainable params
33.2 M    Total params
13

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 7: 100%|██████████| 250/250 [00:56<00:00,  4.45it/s, v_num=10, val_acc=0.431, train_loss=0.742, train_acc=0.526]
Best checkpoint saved to: /home/turbotowerlnx/Documents/Master/ARA/ARA-Medical-MissingData-Robust-Model/models/fast_resnet3d_best-v5.ckpt


FastResNet3DLightningModule(
  (model): VideoResNet(
    (stem): BasicStem(
      (0): Conv3d(4, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
      (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Sequential(
          (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
          (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (conv2): Sequential(
          (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
          (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (relu): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Sequential(
          (0): Conv3DSimple(64, 64, ke

# Test

In [ ]:
# Test the best checkpoint saved during training
test_metrics = test_fast_resnet(
    config=config,
    batch_size=16,
    num_workers=2,
    checkpoint_path=None,  # auto-picks latest fast_resnet3d_best*.ckpt
)

test_metrics

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/turbotowerlnx/miniconda3/envs/ARA_env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Testing DataLoader 0: 100%|██████████| 26/26 [00:05<00:00,  4.97it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.5098039507865906
        test_loss           1.7765111923217773
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Tested checkpoint: ../models/fast_resnet3d_best-v5.ckpt
Test metrics: [{'test_loss': 1.7765111923217773, 'test_acc': 0.5098039507865906}]


[{'test_loss': 1.7765111923217773, 'test_acc': 0.5098039507865906}]

In [12]:
import os
from glob import glob
import pandas as pd
import torch
from torchmetrics.classification import MulticlassF1Score

from src.config import Configuration
from src.data import UPennGBMDataset
from src.models import FastResNet3DLightningModule

config = Configuration()
test_ds = UPennGBMDataset(config, partition="test")

pattern = os.path.join(config.MODELS_PATH, "fast_resnet3d_best*.ckpt")
candidates = glob(pattern)
if not candidates:
    raise FileNotFoundError(f"No checkpoint found matching: {pattern}")
checkpoint_path = max(candidates, key=os.path.getmtime)

device = torch.device("cuda" if torch.cuda.is_available() and config.cuda else "cpu")
model = FastResNet3DLightningModule.load_from_checkpoint(checkpoint_path).to(device)
model.eval()

rows = []
y_true = []
y_pred = []

with torch.no_grad():
    for idx in range(len(test_ds)):
        sample_i = test_ds[idx]
        x_i = sample_i["image"].unsqueeze(0).to(device)
        true_label = int(sample_i["label"].item())

        logits_i = model(x_i)
        pred_label = int(torch.argmax(logits_i, dim=1).item())

        rows.append(
            {
                "patient": str(test_ds.df.iloc[idx]["ID"]),
                "original_label": true_label,
                "predicted_label": pred_label,
                "fscore": float(true_label == pred_label),
            }
        )
        y_true.append(true_label)
        y_pred.append(pred_label)

results_df = pd.DataFrame(rows).sort_values("patient").reset_index(drop=True)

metric_macro = MulticlassF1Score(num_classes=len(config.labels), average="macro")
metric_weighted = MulticlassF1Score(num_classes=len(config.labels), average="weighted")
y_true_t = torch.tensor(y_true)
y_pred_t = torch.tensor(y_pred)
overall_macro_f1 = metric_macro(y_pred_t, y_true_t).item()
overall_weighted_f1 = metric_weighted(y_pred_t, y_true_t).item()

display(results_df)
print(f"Checkpoint: {checkpoint_path}")
print(f"Overall macro F1: {overall_macro_f1:.4f}")
print(f"Overall weighted F1: {overall_weighted_f1:.4f}")

,patient,original_label,predicted_label,fscore
0,UPENN-GBM-00003,1,0,0.0
1,UPENN-GBM-00011,1,0,0.0
2,UPENN-GBM-00026,0,0,1.0
3,UPENN-GBM-00031,1,0,0.0
4,UPENN-GBM-00032,0,0,1.0
5,UPENN-GBM-00057,1,0,0.0
6,UPENN-GBM-00065,0,0,1.0
7,UPENN-GBM-00075,0,0,1.0
8,UPENN-GBM-00080,1,0,0.0
9,UPENN-GBM-00081,0,0,1.0


Checkpoint: ../models/fast_resnet3d_best-v5.ckpt
Overall macro F1: 0.3377
Overall weighted F1: 0.3443
